# apertus-eval-prep — vLLM on Colab

[Open in Colab](https://colab.research.google.com/github/Shivani767/apertus-eval-prep/blob/master/notebooks/colab_vllm.ipynb)

Runtime → Change runtime type → **T4 GPU**.

This notebook scores the frozen slice with vLLM using **already-rendered** completion prompts (no second chat template). Download the JSON at the end and commit it to `results/vllm_tokenizer.json`.

In [ ]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e .

In [ ]:
# Colab T4: install official vLLM +cu129 wheel (NOT PyPI cu13 default).
import os, sys, subprocess
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Set runtime to GPU (T4) and rerun.'
TORCH_PIN = torch.__version__
VLLM_VER = os.environ.get('APERTUS_VLLM_PIN', '0.27.1')
WHEEL = (
    f'https://github.com/vllm-project/vllm/releases/download/v{VLLM_VER}/'
    f'vllm-{VLLM_VER}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl'
)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'vllm'], check=False)
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', WHEEL,
    '--extra-index-url', 'https://download.pytorch.org/whl/cu128',
])
import torch as _t
if _t.__version__ != TORCH_PIN:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', f'torch=={TORCH_PIN}'])
from vllm import LLM, SamplingParams
import vllm
print(torch.cuda.get_device_name(0), 'torch', TORCH_PIN, 'vllm', getattr(vllm, '__version__', '?'))


In [ ]:
!python -m apertus_eval_prep dump-prompts --config configs/vllm.yaml --out results/prompts_vllm.txt --n 2
!python -m apertus_eval_prep eval --config configs/vllm.yaml --out results/vllm_tokenizer.json

In [ ]:
# Optional: also run HF generate on the same GPU for a same-hardware backend delta.
!python -m apertus_eval_prep eval --config configs/default.yaml --backend hf --out results/hf_tokenizer_colab.json
!python -m apertus_eval_prep compare results/hf_tokenizer_colab.json results/vllm_tokenizer.json --out results/compare_backend.md
print(open("results/compare_backend.md").read())

In [ ]:
import os
from google.colab import files
for name in [
    "results/vllm_tokenizer.json",
    "results/hf_tokenizer_colab.json",
    "results/compare_backend.md",
]:
    if os.path.exists(name):
        files.download(name)